# Predicting `arrears_next_month`
Follows `protocol.yaml`: temporal holdout (`holdout.csv`), 5-split walk-forward CV on `billing_month` inside the training rows, choice by mean CV ROC AUC, a single holdout score for the chosen candidate.

**Feature decision.** `final_notice_sent` is recorded in the billing cycle *after* the billing month, i.e. after/around the outcome (86.5% of notice accounts are in arrears vs 1.4% otherwise). It is not known at prediction time and is excluded as leakage. `meter_ref` and `billing_month` are excluded per the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET='arrears_next_month'
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='42accdc3f8c6ae3514896d73516d98c54e4a27c5422d4fe88e5c4fe1b00a14f7'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='903bb064736772428ae27c90c6c03ac7f9c28c7eb54a50c70b7601b731258ba6'
df=pd.read_csv('data.csv')
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==20939 and len(hold_idx)==7536
is_hold=df.index.isin(hold_idx)
train=df[~is_hold].copy(); hold=df[is_hold].copy()
print(len(train),len(hold), train.billing_month.max(), hold.billing_month.min())

In [ ]:
CAT=['tariff']; NUM=['occupants','kwh','bill_eur','late_payments_12m']
FEATURES=CAT+NUM   # excluded: meter_ref, billing_month (protocol); final_notice_sent (post-outcome leak)
def prep(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),
                              ('n',StandardScaler() if scale else 'passthrough',NUM)])
# Candidates named before any scoring
CANDIDATES={
 'baseline_prior':Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C1':Pipeline([('p',prep(True)),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'logreg_C0.1':Pipeline([('p',prep(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'random_forest':Pipeline([('p',prep(False)),('m',RandomForestClassifier(n_estimators=300,min_samples_leaf=20,n_jobs=1,random_state=0))]),
 'hist_gb':Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(CANDIDATES))

In [ ]:
# Walk-forward CV by month: months -> 6 consecutive blocks; fold k trains on blocks<=k, validates on block k+1
months=np.sort(train.billing_month.unique())
blocks=np.array_split(months,6)
folds=[]
for k in range(5):
    tr_m=np.concatenate(blocks[:k+1]); va_m=blocks[k+1]
    tr=train.index[train.billing_month.isin(tr_m)].to_numpy()
    va=train.index[train.billing_month.isin(va_m)].to_numpy()
    folds.append((tr,va))
    print(k,len(tr),len(va),va_m[0],va_m[-1])
cv_scores={}
for name,model in CANDIDATES.items():
    s=[]
    for tr,va in folds:
        X_tr,y_tr=train.loc[tr,FEATURES],train.loc[tr,TARGET]
        X_va,y_va=train.loc[va,FEATURES],train.loc[va,TARGET]
        m=clone(model).fit(X_tr,y_tr)   # preprocessing fitted inside the fold
        s.append(float(roc_auc_score(y_va,m.predict_proba(X_va)[:,1])))
    cv_scores[name]=s
cv=pd.DataFrame(cv_scores).T; cv['mean']=cv.mean(axis=1); cv

In [ ]:
chosen=max(cv_scores,key=lambda n:np.mean(cv_scores[n]))
print('chosen by mean CV roc_auc:',chosen)

In [ ]:
# Refit chosen candidate on all training rows; single holdout scoring
final=clone(CANDIDATES[chosen]).fit(train[FEATURES],train[TARGET])
X_ho,y_ho=hold[FEATURES],hold[TARGET]
hold_auc=float(roc_auc_score(y_ho,final.predict_proba(X_ho)[:,1]))
print('HOLDOUT roc_auc (scored once):',round(hold_auc,4))

In [ ]:
evidence={'features':FEATURES,'candidates':list(CANDIDATES),
 'cv_folds':[[[int(i) for i in tr],[int(i) for i in va]] for tr,va in folds],
 'cv_scores':cv_scores,'chosen':chosen,
 'holdout_rows':[int(i) for i in hold.index],
 'holdout_scores':{chosen:{'roc_auc':hold_auc}}}
json.dump(evidence,open('evidence.json','w'))

## Conclusion
See the CV table and holdout ROC AUC above. The holdout score is the headline; it was computed once, after the choice was made on CV alone.